# UACB-604 · Lecture 1.5 — Querying public databases from code

Clicking around websites is fine for one gene. For 500 genes, or for an analysis someone else must be able to repeat, you query databases **from code**.
Every database in this notebook offers a free web API: a URL that returns data instead of a web page.

**You will:**
1. Look up a gene in **NCBI Gene** and discover a coordinate trap
2. Look up the same gene in **Ensembl** and compare
3. See how coordinates change between genome assemblies (GRCh37 → GRCh38)
4. Fetch the protein from **UniProt** and compute its mass yourself
5. List its **KEGG** pathways and see a one-to-many ID mapping
6. Build an ID-mapping table for several genes

Public servers are sometimes slow or briefly down. The helper below retries automatically.

In [ ]:
import json, time, urllib.request
import pandas as pd

def get(url, as_json=True, tries=5):
    for attempt in range(tries):
        try:
            req = urllib.request.Request(url, headers={"Accept": "application/json", "User-Agent": "uacb604-course"})
            text = urllib.request.urlopen(req, timeout=60).read().decode()
            return json.loads(text) if as_json else text
        except Exception as e:
            print(f"  (server said {e}; retrying)")
            time.sleep(2 + 2 * attempt)
    raise RuntimeError("server not responding, try again in a minute: " + url)

EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
SYMBOL = "TP53"

## 1. NCBI Gene
Two calls: **esearch** turns a query into an ID; **esummary** returns a summary for that ID.

In [ ]:
gid = get(f"{EUTILS}/esearch.fcgi?db=gene&term={SYMBOL}[sym]+AND+human[orgn]&retmode=json")["esearchresult"]["idlist"][0]
s = get(f"{EUTILS}/esummary.fcgi?db=gene&id={gid}&retmode=json")["result"][gid]
loc = s["genomicinfo"][0]
print("NCBI Gene ID:", gid, "|", s["description"], "| chromosome", s["chromosome"], s["maplocation"], "| OMIM:", s["mim"])
print("API location:", loc["chraccver"], loc["chrstart"], loc["chrstop"], "exons:", loc["exoncount"])

⚠️ **The coordinate trap.** Open the NCBI Gene page for TP53 in your browser. It shows the location as `NC_000017.11 (7668421..7687490, complement)`.
The API gives `chrstart = 7687489` and `chrstop = 7668420`: **0-based**, and listed in the direction of the gene (minus strand, so start > stop).
The web page is **1-based**. Same database, two conventions.

In [ ]:
lo, hi = sorted([int(loc["chrstart"]), int(loc["chrstop"])])
ncbi_start, ncbi_end = lo + 1, hi + 1           # convert 0-based → 1-based
print(f"NCBI, 1-based: chr17:{ncbi_start:,}-{ncbi_end:,}  (length {ncbi_end - ncbi_start + 1:,} bp)")

## 2. Ensembl
Ensembl's REST API: one URL per question. `expand=1` also returns all transcripts.

In [ ]:
e = get(f"https://rest.ensembl.org/lookup/symbol/homo_sapiens/{SYMBOL}?expand=1")
print("Ensembl ID:", e["id"], "| assembly", e["assembly_name"], f"| chr{e['seq_region_name']}:{e['start']:,}-{e['end']:,}",
      "| strand", e["strand"], "| canonical", e["canonical_transcript"])
tx = pd.DataFrame([{"transcript": t["id"], "biotype": t["biotype"], "start": t["start"], "end": t["end"],
                    "canonical": t.get("is_canonical", 0)} for t in e["Transcript"]])
print(len(tx), "transcripts")
tx["biotype"].value_counts()

### NCBI and Ensembl disagree about where the gene is
Both use GRCh38 and 1-based coordinates here, yet the gene boundaries differ. Each database defines a gene as the span of *its own* set of transcripts.

In [ ]:
print(f"NCBI    : {ncbi_start:,} – {ncbi_end:,}")
print(f"Ensembl : {e['start']:,} – {e['end']:,}")
print(f"Ensembl extends {ncbi_start - e['start']:,} bp further at the low end and {e['end'] - ncbi_end:,} bp at the high end")

## 3. Genome assemblies change coordinates
The reference genome is updated from time to time. GRCh37 (2009) and GRCh38 (2013) put the same gene at different positions.
NCBI keeps the history in `locationhist`.

In [ ]:
hist = pd.DataFrame(s["locationhist"])[["annotationrelease", "assemblyaccver", "chraccver", "chrstart", "chrstop"]]
hist.head(10)

In [ ]:
h37 = hist[hist.assemblyaccver.str.startswith("GCF_000001405.25")].iloc[0]     # GCF_000001405.25 = GRCh37.p13
start37 = min(int(h37.chrstart), int(h37.chrstop)) + 1
print(f"GRCh37: chr17:{start37:,}   GRCh38: chr17:{ncbi_start:,}   shift = {ncbi_start - start37:,} bp")

✏️ **Exercise 1.** A collaborator sends you a variant "chr17:7,577,120". Which assembly is it probably on, and how would you check? Why must every coordinate in a paper state its assembly?

## 4. UniProt: the protein
UniProt's REST API returns the curated (Swiss-Prot) entry, including the sequence.
In Lecture 1.2 we computed peptide masses with *monoisotopic* residue masses. Proteins are usually reported with **average** masses (natural isotope mix). Let's reproduce UniProt's value.

In [ ]:
up = get(f"https://rest.uniprot.org/uniprotkb/search?query=gene_exact:{SYMBOL}+AND+organism_id:9606+AND+reviewed:true&format=json")["results"][0]
acc, seq = up["primaryAccession"], up["sequence"]["value"]
print(acc, "|", up["proteinDescription"]["recommendedName"]["fullName"]["value"], "|", len(seq), "aa | UniProt mass", up["sequence"]["molWeight"], "Da")
print(seq[:60] + "…")

AVG = dict(G=57.0519, A=71.0788, S=87.0782, P=97.1167, V=99.1326, T=101.1051, C=103.1388, L=113.1594, I=113.1594,
           N=114.1038, D=115.0886, Q=128.1307, K=128.1741, E=129.1155, M=131.1926, H=137.1411, F=147.1766,
           R=156.1875, Y=163.1760, W=186.2132)
mass = 18.01528 + sum(AVG[a] for a in seq)
print(f"our calculation: {mass:,.0f} Da")

## 5. KEGG: pathways, and a one-to-many mapping
KEGG names human genes `hsa:<NCBI Gene ID>`. Its API answers with plain tab-separated text.

In [ ]:
kegg = f"hsa:{gid}"
links = get(f"https://rest.kegg.jp/link/pathway/{kegg}", as_json=False).strip().split("\n")
names = dict(l.split("\t") for l in get("https://rest.kegg.jp/list/pathway/hsa", as_json=False).strip().split("\n"))
paths = [l.split("\t")[1].replace("path:", "") for l in links]
print(len(paths), "KEGG pathways contain", SYMBOL)
for p in paths[:8]:
    print(" ", p, names.get(p, ""))
print("\nKEGG → UniProt:")
print(get(f"https://rest.kegg.jp/conv/uniprot/{kegg}", as_json=False))

One KEGG gene maps to **several** UniProt accessions: one reviewed Swiss-Prot entry (`P04637`) plus unreviewed TrEMBL entries.
A mapping between identifiers is a **relation**, not always a function. Always check whether a mapping is one-to-one before you join tables.

## 6. An ID-mapping table
Put it together for several genes. This is a **join** across databases, with the gene symbol as the shared key.

In [ ]:
rows = []
for sym in ["TP53", "HBB", "INS", "CFTR", "BRCA1"]:
    g = get(f"{EUTILS}/esearch.fcgi?db=gene&term={sym}[sym]+AND+human[orgn]&retmode=json")["esearchresult"]["idlist"][0]
    en = get(f"https://rest.ensembl.org/lookup/symbol/homo_sapiens/{sym}?expand=1")
    u = get(f"https://rest.uniprot.org/uniprotkb/search?query=gene_exact:{sym}+AND+organism_id:9606+AND+reviewed:true&fields=accession,length&format=json")["results"][0]
    rows.append({"symbol": sym, "NCBI Gene": g, "Ensembl": en["id"], "chr": en["seq_region_name"],
                 "transcripts": len(en["Transcript"]), "UniProt": u["primaryAccession"], "protein aa": u["sequence"]["length"]})
    time.sleep(0.4)
idmap = pd.DataFrame(rows)
idmap

✏️ **Exercise 2.** Add three genes of your choice to the table. Which has the most Ensembl transcripts? Plot transcripts against protein length. Is there a relationship?

✏️ **Exercise 3 (coordinates).** A BED file line reads `chr17  7668420  7687490  TP53`. BED is 0-based, half-open.
1. Write it in 1-based, fully closed form.
2. Write a function `bed_to_1based(start, end)` and check it gives the same length.
3. Which of NCBI website, NCBI API, Ensembl, BED and Python slicing use which convention?

---
**Further reading:** Pevsner, *Bioinformatics and Functional Genomics*, ch. 2 (access to sequence data and related information) · Buffalo, *Bioinformatics Data Skills*, ch. 9 (working with range data).